# 🌿 AtlasLeaf — Treinamento & Experimentos no Google Colab Pro

Notebook oficial para treinar e comparar receitas do modelo **`field7`** usando GPU do Colab Pro (**T4 / L4 / A100**) com **avaliação honesta** (*camera-holdout* e *cross-dataset*).

### Como usar (resumo rápido)
1. **Runtime → Change runtime type → GPU** (T4 para testes rápidos, L4 ou A100 para rodadas completas).
2. Suba o arquivo `atlasleaf_unified_resized_768.tar.gz` uma única vez para o Google Drive em `MyDrive/AtlasLeaf/`.
3. Rode as células em ordem:
   - Monta o Google Drive e copia/descompacta o dataset no SSD local `/content/` (muito mais rápido que ler direto do Drive).
   - Clona/atualiza o repositório `RafaelMouraG/AtlasLeaf`.
   - Treina salvando checkpoints e relatórios **direto no Google Drive** (seguro mesmo se a sessão cair durante a viagem).
   - Compara experimentos lado a lado e exporta o melhor modelo para `.onnx` + `metadata.json`.

In [ ]:
# 1. Verificar GPU e ambiente CUDA do Colab
import torch
!nvidia-smi
assert torch.cuda.is_available(), "❌ GPU não detectada! Vá em Runtime > Change runtime type > GPU."
print(f"\n✅ PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 2. Montar Google Drive e definir caminhos
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Ajuste se salvou em outra pasta do Drive:
DRIVE_ROOT = Path("/content/drive/MyDrive/AtlasLeaf")
DATASET_TAR = DRIVE_ROOT / "atlasleaf_unified_resized_768.tar.gz"
RUNS_DIR = DRIVE_ROOT / "runs"
EXPORT_DIR = DRIVE_ROOT / "exports"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_TAR.exists(), (
    f"❌ Arquivo não encontrado no Drive: {DATASET_TAR}\n"
    f"   Suba 'atlasleaf_unified_resized_768.tar.gz' para MyDrive/AtlasLeaf/ e rode novamente."
)
print(f"✅ Dataset no Drive: {DATASET_TAR} ({DATASET_TAR.stat().st_size / 1e9:.2f} GB)")
print(f"✅ Checkpoints serão salvos em: {RUNS_DIR}")

In [ ]:
# 3. Clonar/atualizar o repositório AtlasLeaf e instalar apenas extras de ONNX
# (NÃO rodar pip install -r requirements.txt para preservar o PyTorch CUDA do Colab)
REPO_URL = "https://github.com/RafaelMouraG/AtlasLeaf.git"
BRANCH = "main"  # troque se estiver testando outra branch
REPO_DIR = Path("/content/AtlasLeaf")

if not REPO_DIR.exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull --ff-only

!pip install -q onnx onnxruntime onnxscript
!git -C {REPO_DIR} log -n 3 --oneline

In [ ]:
# 4. Copiar e descompactar o dataset no SSD local /content/ (leitura rápida na GPU)
LOCAL_DATA_ROOT = Path("/content/datasets")
LOCAL_DATASET = LOCAL_DATA_ROOT / "unified_resized"

if not (LOCAL_DATASET / "manifest.json").exists():
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print("📦 Extraindo dataset para o SSD local do Colab...")
    !tar -xzf "{DATASET_TAR}" -C "{LOCAL_DATA_ROOT}"
else:
    print("✅ Dataset já extraído no SSD local.")

import json
manifest = json.loads((LOCAL_DATASET / "manifest.json").read_text())
print(f"\n📊 Imagens: {manifest['total_images']} | Fontes: {manifest['datasets_used']}")
for sp in sorted(LOCAL_DATASET.glob("splits_*.json")):
    s = json.loads(sp.read_text())
    print(f"   - {sp.name:36s}: train={len(s['train']):4d} | val={len(s['val']):4d} | test={len(s['test']):4d}")

## 🧪 5. Configurar e rodar um experimento

### Splits disponíveis (todos honestos, sem vazamento)
- `splits_camera.json` — **Baseline oficial**: treina/valida no ASDID Canon, testa no ASDID Moto.
- `splits_camera_mendeley_train.json` — **Multi-fonte**: adiciona Mendeley V2 (reforça *Cercospora*, *Ferrugem* e *Sadia*) ao treino/val e testa no **mesmo holdout Moto** (comparação 1-para-1).
- `splits_mendeley_holdout.json` — **Cross-região duro**: treina no ASDID inteiro e testa só no Mendeley V2.

### Sugestões de experimentos para melhorar o `field7`
1. **Baseline limpo (7 saídas nativas):** `efficientnet_v2_s` + `splits_camera.json` + `--freeze-backbone --domain-aug`.
2. **Efeito da 2ª fonte de campo:** mesma receita usando `splits_camera_mendeley_train.json` (compare o recall de *Cercospora* no Moto!).
3. **Backbones mais fortes:** testar `convnext_tiny` (com `--freeze-backbone` e também sem `--freeze-backbone` usando `--lr 1e-4`).
4. **Batch / GPU:** em **L4 (24 GB)** ou **A100**, `BATCH_SIZE = 32` e `ACCUM_STEPS = 2` (batch efetivo = 64) + `USE_AMP = True` voa!

In [ ]:
# 5. Parâmetros do experimento (altere EXP_NAME a cada teste para não sobrescrever!)
EXP_NAME = "exp01_effv2s_frozen_camera_mendeley"
MODEL_NAME = "efficientnet_v2_s"       # efficientnet_v2_s | convnext_tiny | efficientnet_b3 | resnet50 | mobilenet_v3
SPLIT_FILE = "splits_camera_mendeley_train.json"  # splits_camera.json | splits_camera_mendeley_train.json | splits_mendeley_holdout.json
EPOCHS = 40
BATCH_SIZE = 32                        # 32 para L4/A100; reduza para 16 na T4 se der OOM com backbone descongelado
ACCUM_STEPS = 2                        # batch efetivo = BATCH_SIZE * ACCUM_STEPS
LR = 5e-4                              # use ~1e-4 se FREEZE_BACKBONE = False
FREEZE_BACKBONE = True
DOMAIN_AUG = True
USE_AMP = True                         # mixed precision BF16/FP16 na GPU
NUM_WORKERS = 4
TARGET_PRECISION = 0.90

CKPT_OUT = RUNS_DIR / f"{EXP_NAME}.pth"

cmd = [
    "python", "scripts/train_field7.py",
    "--data-dir", str(LOCAL_DATASET),
    "--source-split",
    "--split-file", SPLIT_FILE,
    "--field7",
    "--model", MODEL_NAME,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--accumulation-steps", str(ACCUM_STEPS),
    "--lr", str(LR),
    "--num-workers", str(NUM_WORKERS),
    "--target-precision", str(TARGET_PRECISION),
    "--ckpt-out", str(CKPT_OUT),
]
if FREEZE_BACKBONE:
    cmd.append("--freeze-backbone")
if DOMAIN_AUG:
    cmd.append("--domain-aug")
if USE_AMP:
    cmd.append("--amp")

import subprocess
print("🚀 Comando:\n  " + " ".join(cmd) + "\n")
proc = subprocess.run(cmd, cwd=str(REPO_DIR / "Back"))
assert proc.returncode == 0, f"❌ Treino falhou com código {proc.returncode}"
print(f"\n✅ Checkpoint salvo no Drive: {CKPT_OUT}")

In [ ]:
# 6. Comparar todos os experimentos salvos no Google Drive
import torch
import pandas as pd

rows = []
for pth in sorted(RUNS_DIR.glob("*.pth")):
    ck = torch.load(pth, map_location="cpu", weights_only=False)
    cfg = ck.get("config", {})
    run = ck.get("run", {})
    test_m = ck.get("test_metrics", {}).get("test", {})
    sel = test_m.get("selective", {})
    cerc_rec = test_m.get("per_class", {}).get("cercospora_blight", {}).get("recall")
    rows.append({
        "exp": pth.stem,
        "model": cfg.get("model_name", "?"),
        "split": run.get("split_file", "?"),
        "epoch": (ck.get("epoch") + 1) if isinstance(ck.get("epoch"), int) else None,
        "val_bal_acc": round(ck.get("val_bal_acc", 0.0), 2),
        "test_bal_acc": round(test_m.get("test_balanced_acc", 0.0), 2) if test_m else None,
        "test_acc": round(test_m.get("test_accuracy", 0.0), 2) if test_m else None,
        "cercospora_recall": round(cerc_rec * 100, 1) if cerc_rec is not None else None,
        "thr": round(sel.get("threshold", 0.0), 3) if sel else None,
        "cov_%": round(sel.get("coverage", 0.0) * 100, 1) if sel else None,
        "sel_acc_%": round(sel.get("accuracy", 0.0), 1) if sel and sel.get("accuracy") is not None else None,
    })

df = pd.DataFrame(rows)
if len(df):
    display(df.sort_values("test_bal_acc", ascending=False))
else:
    print("Nenhum checkpoint encontrado em", RUNS_DIR)

In [ ]:
# 7. Exportar o melhor experimento para ONNX + metadata.json no Google Drive
BEST_EXP = EXP_NAME  # troque pelo nome do melhor experimento da tabela acima
BEST_CKPT = RUNS_DIR / f"{BEST_EXP}.pth"
OUT_DIR = EXPORT_DIR / BEST_EXP
OUT_DIR.mkdir(parents=True, exist_ok=True)

!cd {REPO_DIR}/Back && python scripts/export_model.py \
    --ckpt "{BEST_CKPT}" \
    --onnx-out "{OUT_DIR / 'model.onnx'}" \
    --meta-out "{OUT_DIR / 'metadata.json'}"

# Verificação rápida no ONNX Runtime
import numpy as np, onnxruntime as ort
sess = ort.InferenceSession(str(OUT_DIR / "model.onnx"), providers=["CPUExecutionProvider"])
out = sess.run(None, {"input": np.zeros((1, 3, 384, 384), dtype=np.float32)})[0]
print(f"\n✅ ONNX validado! Output shape={out.shape} | Arquivos em {OUT_DIR}:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"   - {f.name} ({f.stat().st_size / 1e6:.2f} MB)")